In [1]:
import sagemaker, boto3, json
from sagemaker.model import Model

role = "arn:aws:iam::371087393859:role/defaultrole"
bucket = "ir-sagemaker"

session = boto3.Session(profile_name="lprofile", region_name="us-east-1")

sm_session = sagemaker.Session(boto_session=session, default_bucket=bucket)
region = sm_session.boto_region_name
container_version = '0.33.0-lmi15.0.0-cu128'

container_uri = f'763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{container_version}'
instance_type = "ml.g5.12xlarge"
endpoint_name = ""

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/xdg-ubuntu/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/lbrenap/.config/sagemaker/config.yaml


/home/lbrenap/miniconda3/envs/legalpacaenv/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [ ]:

env = {
    "HF_MODEL_ID": "hltcoe/Rank-K-32B",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_ROLLING_BATCH": "disable",
    "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
    "TENSOR_PARALLEL_DEGREE": "max",
}

model = Model(
    image_uri=container_uri,
    role=role,
    env=env,
    sagemaker_session=sm_session,
)

endpoint_name = sagemaker.utils.name_from_base("rankk-vllm")
print(endpoint_name)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=instance_type,
    endpoint_name=endpoint_name,
    container_startup_health_check_timeout = 1800,
)

print("InService as:", endpoint_name)


In [ ]:
if not endpoint_name:
    endpoint_name = "rankk-vllm-2025-09-26-15-11-35-617"
smr_client = boto3.client('sagemaker-runtime')

desc = sm_session.describe_endpoint(endpoint_name)
print("Status:", desc["EndpointStatus"])
print("ARN:", desc["EndpointArn"])

body = {
    "messages": [
        {"role": "user", "content": "Name places to visit in the US"}
    ],
    "temperature": 0.7,
    "max_tokens": 256,
    "stream": True,
}

resp = smr_client.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    Body=json.dumps(body),
    ContentType='application/json',
)

In [ ]:
print("Response:", end=' ', flush=True)
full_response = ""

for event in resp['Body']:
    if 'PayloadPart' in event:
        chunk = event['PayloadPart']['Bytes'].decode()

        try:
            if chunk.startswith('data: '):
                data = json.loads(chunk[6:])  # Skip "data: " prefix
            else:
                data = json.loads(chunk)

            if 'choices' in data and len(data['choices']) > 0:
                if 'delta' in data['choices'][0] and 'content' in data['choices'][0]['delta']:
                    token_text = data['choices'][0]['delta']['content']
                    full_response += token_text
                    print(token_text, end='', flush=True)

        except json.JSONDecodeError:
            continue

In [1]:
rank_k_prompt = """
Determine a ranking of the passages based on how well they replace the masked passage in the argument chain.
How relevant a passage is depends on how well it entails the posterior passage and how well is entailed by the preceding one.
Sort them from the most relevant to the least.
Answer with the passage number using a format of '[3] > [2] > [4] = [1] > [5].
Ties are acceptable if they are equally relevant.
You need you to be accurate. Don't think, just give the sorting output in the described format.
Output only the ordering with no other text.

Query: {query}

{docs}
"""

In [ ]:
from reranking_pipeline import (
    _read_streaming_body,
    _call_llm_rank_order,
    _fill_rankk_template,
    rerank_topk50_rankk_all,
)


In [2]:
import os
import importlib
import reranking_pipeline as rp
from pathlib import Path

rp = importlib.reload(rp)
# rank_provider = os.environ.get('RERANK_PROVIDER', 'sagemaker').lower()
rank_provider = 'openai'
openai_client = None
openai_model = 'gpt-5-mini'

if rank_provider == 'openai':
    from openai import OpenAI
    openai_client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
    smr_client = None
    endpoint_name = None
else:
    smr_client = boto3.client('sagemaker-runtime')

print('Using provider:', rank_provider, '| pipeline version:', rp.PIPELINE_VERSION)


Using provider: openai | pipeline version: 2025-09-30


In [ ]:
# processes every query present in topk_50.jsonl
rp.rerank_topk50_rankk_all(
    smr_client=smr_client,
    endpoint_name=endpoint_name,
    rank_k_prompt=rank_k_prompt,
    provider=rank_provider,
    openai_client=openai_client,
    openai_model=openai_model,
    topk_path=Path("retrieval_results/retrieved/topk_50.jsonl"),
    out_dir=Path("retrieval_results/reranked"),
    window_k=8, stride=6, snippet_limit=480,
    temperature=1, max_tokens=4000,
    run_name="rankk_top50",
    verbose=False,
    resume=True,
    processing_batch_size = 600,
)

# The final files will be in retrieval_results/reranked/


In [4]:
metrics = rp.evaluate_reranked_jsonl(
    Path('retrieval_results/reranked/rankk_top50.jsonl'),
    ks=(1, 5, 10, 20, 50),
    verbose_examples=5,
)
metrics



=== Macro metrics (from RERANKED top-k) ===
Hit@ 1 = 0.0559
Hit@ 5 = 0.1366
Hit@10 = 0.3623
Hit@20 = 0.5135
Hit@50 = 0.6729
MRR     = 0.1275
Mean rank (found) = 14.06
Queries with positive in top-k = 325 of 483

=== Examples ===
- R2011_France Télécom SA v European Commission:0 | doc=None | pos_rank=None | found={1: False, 5: False, 10: False, 20: False, 50: False}
  top3: ['R2011_France Télécom SA v European Commission:203', 'R2011_France Télécom SA v European Commission:54', 'R2011_France Télécom SA v European Commission:153']
- R2011_France Télécom SA v European Commission:1 | doc=None | pos_rank=None | found={1: False, 5: False, 10: False, 20: False, 50: False}
  top3: ['R2011_France Télécom SA v European Commission:202', 'R2011_France Télécom SA v European Commission:166', 'R2011_France Télécom SA v European Commission:93']
- R2011_France Télécom SA v European Commission:2 | doc=R2011_France Télécom SA v European Commission | pos_rank=9 | found={1: False, 5: False, 10: True, 20: 

{'num_queries': 483,
 'hit_rate': {1: 0.055900621118012424,
  5: 0.13664596273291926,
  10: 0.36231884057971014,
  20: 0.5134575569358178,
  50: 0.6728778467908902},
 'mrr': 0.1275089851168986,
 'mean_rank_found': 14.055384615384616,
 'queries_with_positive_in_topk': 325,
 'per_doc': {None: {'num_queries': 158, 'avg_rank': nan, 'mrr': nan},
  'R2011_France Télécom SA v European Commission': {'num_queries': 78,
   'avg_rank': 9.076923076923077,
   'mrr': 0.263554082877136},
  'R2011_European Commission v Kronoply GmbH & Co': {'num_queries': 74,
   'avg_rank': 17.513513513513512,
   'mrr': 0.12453418175474708},
  'R2016_DTS Distribuidora de Televisión Digital': {'num_queries': 83,
   'avg_rank': 13.650602409638553,
   'mrr': 0.2010892042196909},
  'A2017_European Commission v Italian Republic_DT': {'num_queries': 90,
   'avg_rank': 15.9,
   'mrr': 0.1680409771884398}}}

In [ ]:
print(f"Deleting SageMaker resources for endpoint: {endpoint_name}")
sm_session.delete_endpoint(endpoint_name)
sm_session.delete_endpoint_config(endpoint_name)


In [ ]:
Hit@ 1 = 0.0045
Hit@ 5 = 0.0090
Hit@10 = 0.1171
Hit@20 = 0.7072
Hit@50 = 0.9730

In [11]:
hit1 = 0.0045
hit5 = 0.0090
hit10 = 0.1171
hit20 = 0.7072
hit50 = 0.9730
0.6729*(0.7072)

0.47587488000000006

In [2]:
from pathlib import Path
import reranking_pipeline as rp

metrics = rp.evaluate_doc_full_recall_jsonl(
    Path('retrieval_results/reranked/rankk_top50.jsonl'),
    ks=(1, 5, 10, 20, 50),
)
metrics


=== Document coverage (all queries within top-k) ===
Docs Hit@ 1 (all queries): 0/4 (0.00%)
Docs Hit@ 5 (all queries): 0/4 (0.00%)
Docs Hit@10 (all queries): 0/4 (0.00%)
Docs Hit@20 (all queries): 0/4 (0.00%)
Docs Hit@50 (all queries): 0/4 (0.00%)


{'num_documents': 4,
 'docs_with_all_queries_in_topk': {1: 0, 5: 0, 10: 0, 20: 0, 50: 0},
 'docs_with_all_queries_in_topk_pct': {1: 0.0,
  5: 0.0,
  10: 0.0,
  20: 0.0,
  50: 0.0},
 'queries_per_doc': {'R2011_France Télécom SA v European Commission': 144,
  'R2011_European Commission v Kronoply GmbH & Co': 88,
  'R2016_DTS Distribuidora de Televisión Digital': 148,
  'A2017_European Commission v Italian Republic_DT': 103}}

In [ ]:
a -> b -> c -> d
Whether c appears in the top k for d

a -> b
c -> b
whether at top k a and c appear